# Project 1B: Real Word2Vec Training (WikiText)
Train Skip-Gram Word2Vec on real WikiText, **and inspect the actual shallow ANN** with TensorFlow: embedding matrices, forward pass, negative sampling, BCE loss, gradients, and backpropagation. Gensim trains the full selected corpus; the transparent TensorFlow ANN uses a smaller sample for teaching.

In [ ]:
!pip -q install 'gensim>=4.3.3' 'datasets>=3.0' scikit-learn matplotlib

## 1. Settings
Switch to `wikitext-103-raw-v1` for a larger training corpus.

In [ ]:
DATASET_SIZE='wikitext-2-raw-v1' # or 'wikitext-103-raw-v1'
DIM=100
WINDOW=5
MIN_COUNT=3 if '2-' in DATASET_SIZE else 5
EPOCHS=10 if '2-' in DATASET_SIZE else 5
SEED=42

## 2. Download and tokenize the Wikipedia-derived training text

In [ ]:
from datasets import load_dataset
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np,time
raw=load_dataset('Salesforce/wikitext',DATASET_SIZE,split='train')
def tokenize(line):
    if not line.strip() or line.lstrip().startswith('='):return []
    return simple_preprocess(line,deacc=True,min_len=2,max_len=25)
sentences=[t for s in raw['text'] if len(t:=tokenize(s))>=3]
print('Sentences:',len(sentences),'Tokens:',sum(map(len,sentences)))
print('Example:',sentences[0][:25])

## 3. Train Skip-Gram with Negative Sampling
`sg=1` means Skip-Gram, `negative=10` means ten sampled negatives. Track the approximate training loss after each epoch.

In [ ]:
model=Word2Vec(vector_size=DIM,window=WINDOW,min_count=MIN_COUNT,sg=1,negative=10,hs=0,workers=2,seed=SEED,compute_loss=True)
model.build_vocab(sentences)
print('Vocabulary:',len(model.wv))
losses=[]
start=time.time()
for epoch in range(EPOCHS):
    alpha0=0.025+(0.001-0.025)*epoch/EPOCHS
    alpha1=0.025+(0.001-0.025)*(epoch+1)/EPOCHS
    model.train(sentences,total_examples=model.corpus_count,epochs=1,start_alpha=alpha0,end_alpha=alpha1,compute_loss=True)
    # get_latest_training_loss() is the loss reported for the latest train() call.
    loss=model.get_latest_training_loss()
    losses.append(loss)
    print(f'Epoch {epoch+1}/{EPOCHS}: approximate loss={loss:,.0f}; elapsed={time.time()-start:.0f}s')
plt.plot(range(1,EPOCHS+1),losses,marker='o');plt.xlabel('Epoch');plt.ylabel('Approximate loss');plt.grid(alpha=.3);plt.show()

## 4. Nearest neighbors and cosine similarity
Small corpora can produce noisy relationships. Inspect results rather than assuming they will be meaningful.

In [ ]:
for word in ['king','queen','computer','science','city','music']:
    if word in model.wv:print(word,model.wv.most_similar(word,topn=6))
for a,b in [('king','queen'),('cat','dog'),('science','computer')]:
    if a in model.wv and b in model.wv:print(a,b,round(float(model.wv.similarity(a,b)),3))

## 3A. What ANN does Word2Vec actually train?

**Skip-Gram with Negative Sampling (SGNS)** is a shallow neural network. The center word ID selects a row of `W_in` (the **input embedding matrix**). A neighboring word ID selects a row of `W_out` (the **context embedding matrix**). Their dot product passes through a sigmoid to predict whether this word pair is a real neighboring pair (label **1**) or a randomly sampled negative pair (label **0**).

`center ID → Embedding(W_in) ─┐`

`                              dot product → sigmoid → BCE loss`

`context ID → Embedding(W_out) ┘`

**Backpropagation updates both matrices.** There is *no* extra dense nonlinear hidden layer and *no* full-vocabulary softmax in SGNS. Gensim's `model.train()` implements this efficiently in compiled code; the TensorFlow section below shows the same architecture explicitly. The small TensorFlow run is for inspection, while Gensim remains the large-corpus training.

In [ ]:
import tensorflow as tf
import numpy as np

tf.random.set_seed(SEED)
# Use a small subset of WikiText sentences for the transparent ANN demonstration.
# Both branches are trainable lookup tables: IDs -> 100-dimensional vectors.
VOCAB_SIZE = len(model.wv)
word_to_id = model.wv.key_to_index

class VisibleSkipGram(tf.keras.Model):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.input_embedding = tf.keras.layers.Embedding(vocab_size, dim, name='W_in')
        self.output_embedding = tf.keras.layers.Embedding(vocab_size, dim, name='W_out')

    def call(self, inputs):
        center_ids, context_ids = inputs
        center_vectors = self.input_embedding(center_ids)
        context_vectors = self.output_embedding(context_ids)
        logits = tf.reduce_sum(center_vectors * context_vectors, axis=-1)
        return logits  # sigmoid is included in BinaryCrossentropy(from_logits=True)

ann = VisibleSkipGram(VOCAB_SIZE, DIM)
_ = ann((tf.constant([0]), tf.constant([1])))  # build weights
ann.summary()
print('W_in shape:', ann.input_embedding.embeddings.shape)
print('W_out shape:', ann.output_embedding.embeddings.shape)
print('Total trainable parameters:', sum(np.prod(v.shape) for v in ann.trainable_variables))

## 3B. Construct positive and negative word pairs from real WikiText
This simplified mini-batch generator samples nearby context words and random negatives. It is **not** Gensim's frequency-adjusted negative-sampling distribution; it is designed to make the learning calculation visible. We cap the data for Colab speed. The preceding Gensim training still uses all selected WikiText sentences.

In [ ]:
import random
rng = np.random.default_rng(SEED)
positive_pairs=[]
for sentence in sentences[:5000]:
    token_ids=[word_to_id[w] for w in sentence if w in word_to_id]
    for pos,center_id in enumerate(token_ids):
        for j in range(max(0,pos-WINDOW),min(len(token_ids),pos+WINDOW+1)):
            if j!=pos: positive_pairs.append((center_id,token_ids[j]))
            if len(positive_pairs)>=45000: break
        if len(positive_pairs)>=45000:break
    if len(positive_pairs)>=45000:break
assert positive_pairs, 'No word pairs created; try a lower MIN_COUNT.'
positive_pairs=np.array(positive_pairs,dtype=np.int32)
neg_context=rng.integers(0,VOCAB_SIZE,size=len(positive_pairs),dtype=np.int32)
pair_centers=np.concatenate([positive_pairs[:,0],positive_pairs[:,0]])
pair_contexts=np.concatenate([positive_pairs[:,1],neg_context])
pair_labels=np.concatenate([np.ones(len(positive_pairs)),np.zeros(len(positive_pairs))]).astype('float32')
dataset=tf.data.Dataset.from_tensor_slices(((pair_centers,pair_contexts),pair_labels)).shuffle(min(len(pair_labels),30000),seed=SEED).batch(512).prefetch(tf.data.AUTOTUNE)
print('Positive samples:',len(positive_pairs),'Negative samples:',len(neg_context))
print('Example positive:',model.wv.index_to_key[int(positive_pairs[0,0])],model.wv.index_to_key[int(positive_pairs[0,1])])

## 3C. Forward pass → loss → gradients → weight update

For a positive pair, `label=1`; for a negative pair, `label=0`. We use binary cross-entropy on raw dot-product logits. A `GradientTape` measures how changing the embedding values would change the loss; Adam applies the gradients. **This is the actual ANN training loop**, not a pre-trained model. To make its effect observable, compare the same input-vector row before and after training.

In [ ]:
optimizer=tf.keras.optimizers.Adam(learning_rate=0.01)
loss_fn=tf.keras.losses.BinaryCrossentropy(from_logits=True)
probe_id=word_to_id.get('king',0)
weights_before=ann.input_embedding.embeddings.numpy()[probe_id].copy()
ann_losses=[]
for epoch in range(4):
    epoch_losses=[]
    for (centers,contexts),labels in dataset:
        with tf.GradientTape() as tape:
            logits=ann((centers,contexts),training=True) # FORWARD
            loss=loss_fn(labels,logits)                   # LOSS
        gradients=tape.gradient(loss,ann.trainable_variables) # BACKPROP
        optimizer.apply_gradients(zip(gradients,ann.trainable_variables)) # UPDATE
        epoch_losses.append(float(loss))
    ann_losses.append(np.mean(epoch_losses))
    print(f'TensorFlow ANN epoch {epoch+1}: BCE loss {ann_losses[-1]:.4f}')
weights_after=ann.input_embedding.embeddings.numpy()[probe_id]
print('Example word:',model.wv.index_to_key[probe_id])
print('W_in first 5 values BEFORE:',np.round(weights_before[:5],4))
print('W_in first 5 values AFTER: ',np.round(weights_after[:5],4))
print('Change in vector (L2 norm):',round(float(np.linalg.norm(weights_after-weights_before)),4))
plt.figure(figsize=(6,3));plt.plot(range(1,len(ann_losses)+1),ann_losses,marker='o');plt.title('Visible TensorFlow ANN training loss');plt.xlabel('Epoch');plt.ylabel('Binary cross-entropy');plt.grid(alpha=.25);plt.show()

## 3D. Inspect a prediction and compare with Gensim
The sigmoid of a dot product is the ANN's learned score for whether two words co-occur within the training window. It is **not** a calibrated probability of overall semantic relatedness. Gensim and TensorFlow are trained separately here, so their coordinates need not match.

In [ ]:
for left,right in [('king','queen'),('cat','dog'),('science','computer')]:
    if left in word_to_id and right in word_to_id:
        a=tf.constant([word_to_id[left]]);b=tf.constant([word_to_id[right]])
        logit=ann((a,b),training=False)
        print(f'{left:>10s} + {right:<10s} | ANN sigmoid score: {float(tf.sigmoid(logit)[0]):.3f} | Gensim cosine: {float(model.wv.similarity(left,right)):.3f}')

## 5. Vector arithmetic
Analogies like king − man + woman ≈ queen are **not guaranteed**, especially with WikiText-2.

In [ ]:
def analogy(positive,negative):
    missing=[w for w in positive+negative if w not in model.wv]
    if missing:print('Missing words:',missing)
    else:print('Analogy:',model.wv.most_similar(positive=positive,negative=negative,topn=5))
analogy(['king','woman'],['man'])
analogy(['paris','germany'],['france'])

## 6. Visualize 100D vectors in 2D with PCA
PCA preserves variance, not exact semantic distances.

In [ ]:
words=[w for w in ['king','queen','man','woman','dog','cat','animal','computer','software','science','music','song','city','country','paris','france'] if w in model.wv]
if len(words)>=3:
    xy=PCA(n_components=2).fit_transform(np.array([model.wv[w] for w in words]))
    plt.figure(figsize=(10,7));plt.scatter(xy[:,0],xy[:,1])
    for word,(x,y) in zip(words,xy):plt.annotate(word,(x,y),xytext=(5,5),textcoords='offset points')
    plt.title('Learned Word2Vec embeddings — PCA');plt.grid(alpha=.2);plt.show()

## 7. Save the trained model and vectors

In [ ]:
model.save('wikitext_word2vec.model')
model.wv.save('wikitext_word_vectors.kv')
print('Saved model and vectors. Download via Colab Files sidebar.')

## Student challenges
1. Compare WikiText-2 and WikiText-103.
2. Change vector dimension 100 → 50 or 200.
3. Change context window 5 → 2 or 10.
4. Compare neighbors before and after more training epochs.
5. Why does training loss not guarantee a successful analogy?

**Notes:** Training time varies by Colab hardware. Results vary across runs with multiworker training.